<a href="https://colab.research.google.com/github/fortidatasolutions-ctrl/train/blob/main/DanMetadata_Writer_tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##### Copyright 2024 The AI Edge Authors.

### Creating and Activating a Virtual Environment in Colab (Demonstration)

While Colab provides an isolated environment for each session, you can still create and use a virtual environment within it if you need to. Note that the virtual environment will be created in the current working directory (`/content/`).

In [ ]:
# Create a virtual environment named '.venv'
!python -m venv .venv

print("Ensuring pip and setuptools are installed and upgraded in the virtual environment...")
import os
venv_python_executable = "./.venv/bin/python"
venv_pip_executable = "./.venv/bin/pip"

# Check if pip executable exists. If not, install it via get-pip.py
# This handles cases where python -m venv fails to install pip properly due to `ensurepip` issues.
if not os.path.exists(venv_pip_executable):
    print("pip executable not found in .venv. Attempting manual installation with get-pip.py...")
    # Download get-pip.py into the content directory
    !curl -sSL https://bootstrap.pypa.io/get-pip.py -o get-pip.py
    # Install pip into the virtual environment using its python interpreter
    !{venv_python_executable} get-pip.py
    # Clean up get-pip.py
    !rm get-pip.py
    print("pip installed in .venv using get-pip.py.")
else:
    print("pip executable found in .venv.")

# Now that pip should definitely be present, ensure it's upgraded along with setuptools
print("Upgrading pip and setuptools in the virtual environment...")
!{venv_python_executable} -m pip install --upgrade pip setuptools

print("Virtual environment created and pip/setuptools ensured: .venv")

In [ ]:
# To use the virtual environment's Python interpreter and install packages:
# We will use the full path to the python executable within the venv.
!/content/.venv/bin/pip install pandas

print("Pandas installed in the virtual environment.")

# Verify that pandas is available in this specific environment
!/content/.venv/bin/python -c "import pandas; print(pandas.__version__)"

Please note that the activation of a virtual environment using `source` commands in `!shell` commands within Colab cells usually only applies to that specific cell's subshell. For persistent use across multiple cells, you would typically refer to the full path of the Python executable and `pip` within the `.venv` (e.g., `!/content/.venv/bin/python` or `!/content/.venv/bin/pip`).

In [ ]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# TensorFlow Lite Metadata Writer API



<table class="tfo-notebook-buttons" align="left">
  <td>
    <a target="_blank" href="https://www.tensorflow.org/lite/models/convert/metadata_writer_tutorial"><img src="https://www.tensorflow.org/images/tf_logo_32px.png" />View on TensorFlow.org</a>
  </td>
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/tensorflow/tensorflow/blob/master/tensorflow/lite/g3doc/models/convert/metadata_writer_tutorial.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
  </td>
  <td>
    <a target="_blank" href="https://github.com/tensorflow/tensorflow/blob/master/tensorflow/lite/g3doc/models/convert/metadata_writer_tutorial.ipynb"><img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />View source on GitHub</a>
  </td>
  <td>
    <a href="https://storage.googleapis.com/tensorflow_docs/tensorflow/tensorflow/lite/g3doc/models/convert/metadata_writer_tutorial.ipynb"><img src="https://www.tensorflow.org/images/download_logo_32px.png" />Download notebook</a>
  </td>

</table>

[TensorFlow Lite Model Metadata](https://www.tensorflow.org/lite/models/convert/metadata) is a standard model description format. It contains rich semantics for general model information, inputs/outputs, and associated files, which makes the model self-descriptive and exchangeable.

Model Metadata is currently used in the following two primary use cases:
1. **Enable easy model inference using TensorFlow Lite [Task Library](https://www.tensorflow.org/lite/inference_with_metadata/task_library/overview) and [codegen tools](https://www.tensorflow.org/lite/inference_with_metadata/codegen)**. Model Metadata contains the mandatory information required during inference, such as label files in image classification, sampling rate of the audio input in audio classification, and tokenizer type to process input string in Natural Language models.

2. **Enable model creators to include documentation**, such as the description of model inputs/outputs or how to use the model. Model users can view this documentation via visualization tools such as [Netron](https://netron.app/).

TensorFlow Lite Metadata Writer API provides an easy-to-use API to create Model Metadata for popular ML tasks supported by the TFLite Task Library.  This notebook shows examples of how the metadata should be populated for the following tasks below:

*   [Image classifiers](#image_classifiers)
*   [Object detectors](#object_detectors)
*   [Image segmenters](#image_segmenters)
*   [Natural language classifiers](#nl_classifiers)
*   [Audio classifiers](#audio_classifiers)

Metadata writers for BERT natural language classifiers and BERT question answerers are coming soon.

If you want to add metadata for use cases that are not supported, please use the [Flatbuffers Python API](https://www.tensorflow.org/lite/models/convert/metadata#adding_metadata). See the tutorials [here](https://www.tensorflow.org/lite/models/convert/metadata#adding_metadata).


## Prerequisites

Install the TensorFlow Lite Support Pypi package.

In [ ]:
# Use the python executable from the virtual environment to install mediapipe
!/content/.venv/bin/pip install mediapipe

In [ ]:
# Add the virtual environment's site-packages to sys.path
import sys
import os

venv_site_packages = os.path.join("/content/.venv", "lib", f"python{sys.version_info.major}.{sys.version_info.minor}", "site-packages")

if venv_site_packages not in sys.path:
    sys.path.insert(0, venv_site_packages)
    print(f"Added {venv_site_packages} to sys.path.")
else:
    print(f"{venv_site_packages} is already in sys.path.")

# Verify that tflite_support can now be imported from the venv
try:
    import tflite_support
    print(f"Successfully imported tflite_support from the virtual environment.")
except ImportError:
    print("Failed to import tflite_support. Check your virtual environment setup.")

### Using MediaPipe Tasks

MediaPipe Tasks provides a suite of pre-trained models and tools for common machine learning tasks, especially for on-device applications. Here's a basic example of how to use MediaPipe's Image Classifier.

In [ ]:
# First, let's download a sample image to classify.
!wget -q -O image.jpg https://storage.googleapis.com/mediapipe-assets/cat.jpg

# Import MediaPipe modules
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# Load the pre-trained Image Classifier model (e.g., efficientnet_lite0)
model_path = 'efficientnet_lite0.tflite'
!wget -q -O {model_path} https://storage.googleapis.com/mediapipe-models/image_classifier/efficientnet_lite0/float32/1/efficientnet_lite0.tflite

# Configure the Image Classifier options
base_options = python.BaseOptions(model_asset_path=model_path)
options = vision.ImageClassifierOptions(base_options=base_options, score_threshold=0.25)
classifier = vision.ImageClassifier.create_from_options(options)

# Load the input image
image = mp.Image.create_from_file('image.jpg')

# Classify the image
classification_result = classifier.classify(image)

# Display the results
for category in classification_result.classifications[0].categories:
    print(f"Category: {category.category_name}, Score: {category.score:.2f}")

In [ ]:
# Install tensorflow into the virtual environment
!{venv_pip_executable} install tensorflow

In this example:

1.  We downloaded a sample image (`cat.jpg`) and a pre-trained `efficientnet_lite0.tflite` image classification model.
2.  We imported the necessary `mediapipe.tasks.python` and `mediapipe.tasks.python.vision` modules.
3.  An `ImageClassifier` was created using `create_from_options`, specifying the model path and a score threshold.
4.  The image was loaded using `mp.Image.create_from_file`.
5.  The `classify` method was called on the `classifier` object to get the prediction results.
6.  Finally, the detected categories and their confidence scores were printed.

## Model Training: Fine-tuning MobileNetV2 and Training a New Model

This section will cover the steps to fine-tune an existing `mobilenet_v2` model using transfer learning and also outline the process for training a new model from scratch. We'll start with setting up the TensorFlow environment.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print(f"TensorFlow version: {tf.__version__}")
print(f"Keras version: {keras.__version__}")

In [ ]:
# Define global image dimensions and number of classes for the model
IMG_SIZE = (224, 224)
IMG_SHAPE = IMG_SIZE + (3,)
NUM_CLASSES = 10 # For CIFAR-10 dataset

### Step 1: Prepare the Dataset

Before we can train or fine-tune a model, we need a dataset. For this example, we'll assume a dataset is available for image classification. You would typically load, preprocess, and split your data into training, validation, and test sets here.

In [ ]:
# Placeholder for data loading and preprocessing
# For demonstration, we'll create a dummy dataset or use a standard one like CIFAR-10 or ImageNet subset.

# Example: Load a small dataset for demonstration (e.g., CIFAR-10)
# (Uncomment and run if you want to use CIFAR-10)
# (Note: This will download the dataset if not already present)

(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()
print(f"Raw Training data shape: {x_train.shape}, {y_train.shape}")
print(f"Raw Test data shape: {x_test.shape}, {y_test.shape}")

# Preprocess the data for MobileNetV2
def preprocess_data(images, labels):
    images = tf.image.resize(images, IMG_SIZE) # Resize images to 224x224
    images = keras.applications.mobilenet_v2.preprocess_input(images) # Scale pixels to [-1, 1]
    labels = tf.squeeze(labels) # Remove single-dimensional entries from the shape of an array
    return images, labels

# Create TensorFlow datasets
# Use the number of training samples for shuffle buffer size
SHUFFLE_BUFFER_SIZE = x_train.shape[0]
BATCH_SIZE = 32
PREFETCH_BUFFER_SIZE = tf.data.AUTOTUNE # Use tf.data.AUTOTUNE for optimal buffer size

train_ds = tf.data.Dataset.from_tensor_slices((x_train, y_train))
train_ds = train_ds.map(preprocess_data, num_parallel_calls=tf.data.AUTOTUNE)
train_ds = train_ds.shuffle(SHUFFLE_BUFFER_SIZE).batch(BATCH_SIZE).prefetch(buffer_size=PREFETCH_BUFFER_SIZE)

val_ds = tf.data.Dataset.from_tensor_slices((x_test, y_test))
val_ds = val_ds.map(preprocess_data, num_parallel_calls=tf.data.AUTOTUNE)
val_ds = val_ds.batch(BATCH_SIZE).prefetch(buffer_size=PREFETCH_BUFFER_SIZE)

print(f"Processed Training dataset element spec: {train_ds.element_spec}")
print(f"Processed Validation dataset element spec: {val_ds.element_spec}")

# In a real scenario, you would use `tf.keras.utils.image_dataset_from_directory` or `tf.data.Dataset`
# for more robust and scalable data pipelines.


---

### Step 2: Fine-tuning MobileNetV2

We'll use a pre-trained MobileNetV2 model as a base and add a custom classification head on top. This is a common approach in transfer learning.

In [ ]:
# Create the base model from the pre-trained MobileNetV2 model
base_model = keras.applications.MobileNetV2(input_shape=IMG_SHAPE,
                                           include_top=False, # Don't include the top classification layer
                                           weights='imagenet')

# Freeze the base model to prevent its weights from being updated during the first training phase
base_model.trainable = False

# Create a new model on top
inputs = keras.Input(shape=IMG_SHAPE)
x = keras.applications.mobilenet_v2.preprocess_input(inputs)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)
model = keras.Model(inputs, outputs)

# Compile the model
model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

### Step 2.1: Train the Fine-tuned Model

Now, we'll train the classification head of our fine-tuned `MobileNetV2` model using the prepared dataset. We'll specify the number of epochs and use the validation dataset to monitor performance.

In [ ]:
# Train the model
EPOCHS = 10 # You can adjust this number
history = model.fit(train_ds, epochs=EPOCHS, validation_data=val_ds)

print("Model training complete!")

This sets up the `mobilenet_v2` model for fine-tuning. The `base_model` is frozen initially, and a new classification head is added. The next step would be to train this new head on your specific dataset. After that, you can unfreeze some layers of the `base_model` and fine-tune the entire model with a lower learning rate.

### Step 3: Training a New Model (Alternative/Complementary)

If you prefer to train a new model from scratch, or to complement the fine-tuning approach, you would define your own `Sequential` or `Functional` API model here.

In [ ]:
# Example of defining a simple new model
# (This is illustrative; your actual new model architecture would vary)

# new_model = keras.Sequential([
#     layers.Conv2D(32, (3, 3), activation='relu', input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3)),
#     layers.MaxPooling2D((2, 2)),
#     layers.Conv2D(64, (3, 3), activation='relu'),
#     layers.MaxPooling2D((2, 2)),
#     layers.Conv2D(128, (3, 3), activation='relu'),
#     layers.Flatten(),
#     layers.Dense(64, activation='relu'),
#     layers.Dense(NUM_CLASSES, activation='softmax')
# ])

# new_model.compile(optimizer='adam',
#                   loss='sparse_categorical_crossentropy',
#                   metrics=['accuracy'])

# new_model.summary()

To proceed, you'll need to define how to load and preprocess your specific image dataset (`train_ds`, `val_ds`) for either fine-tuning or training a new model.

## Create Model Metadata for Task Library and Codegen

<a name=image_classifiers></a>
### Image classifiers

See the [image classifier model compatibility requirements](https://www.tensorflow.org/lite/inference_with_metadata/task_library/image_classifier#model_compatibility_requirements) for more details about the supported model format.

Step 1: Import the required packages.

In [ ]:
# from tflite_support.metadata_writers import image_classifier
# from tflite_support.metadata_writers import writer_utils

Step 2: Download the example image classifier, [mobilenet_v2_1.0_224.tflite](https://github.com/tensorflow/tflite-support/raw/master/tensorflow_lite_support/metadata/python/tests/testdata/image_classifier/mobilenet_v2_1.0_224.tflite), and the [label file](https://github.com/tensorflow/tflite-support/raw/master/tensorflow_lite_support/metadata/python/tests/testdata/image_classifier/labels.txt).

In [ ]:
!curl -L https://github.com/tensorflow/tflite-support/raw/master/tensorflow_lite_support/metadata/python/tests/testdata/image_classifier/mobilenet_v2_1.0_224.tflite -o mobilenet_v2_1.0_224.tflite
!curl -L https://github.com/tensorflow/tflite-support/raw/master/tensorflow_lite_support/metadata/python/tests/testdata/image_classifier/labels.txt -o mobilenet_labels.txt

Step 3: Create metadata writer and populate.

In [ ]:
ImageClassifierWriter = image_classifier.MetadataWriter
_MODEL_PATH = "mobilenet_v2_1.0_224.tflite"
# Task Library expects label files that are in the same format as the one below.
_LABEL_FILE = "mobilenet_labels.txt"
_SAVE_TO_PATH = "mobilenet_v2_1.0_224_metadata.tflite"
# Normalization parameters are required when reprocessing the image. It is
# optional if the image pixel values are in the range of [0, 255] and the input
# tensor is quantized to uint8. See the introduction for normalization and
# quantization parameters below for more details.
# https://www.tensorflow.org/lite/models/convert/metadata#normalization_and_quantization_parameters)
_INPUT_NORM_MEAN = 127.5
_INPUT_NORM_STD = 127.5

# Create the metadata writer.
writer = ImageClassifierWriter.create_for_inference(
    writer_utils.load_file(_MODEL_PATH), [_INPUT_NORM_MEAN], [_INPUT_NORM_STD],
    [_LABEL_FILE])

# Verify the metadata generated by the metadata writer.
print(writer.get_metadata_json())

# Populate the metadata into the model.
writer_utils.save_file(writer.populate(), _SAVE_TO_PATH)

<a name=object_detectors></a>
### Object detectors

See the [object detector model compatibility requirements](https://www.tensorflow.org/lite/inference_with_metadata/task_library/object_detector#model_compatibility_requirements) for more details about the supported model format.

Step 1: Import the required packages.

In [ ]:
# from tflite_support.metadata_writers import object_detector
# from tflite_support.metadata_writers import writer_utils

Step 2: Download the example object detector, [ssd_mobilenet_v1.tflite](https://github.com/tensorflow/tflite-support/blob/master/tensorflow_lite_support/metadata/python/tests/testdata/object_detector/ssd_mobilenet_v1.tflite), and the [label file](https://github.com/tensorflow/tflite-support/blob/master/tensorflow_lite_support/metadata/python/tests/testdata/object_detector/labelmap.txt).

In [ ]:
!curl -L https://github.com/tensorflow/tflite-support/raw/master/tensorflow_lite_support/metadata/python/tests/testdata/object_detector/ssd_mobilenet_v1.tflite -o ssd_mobilenet_v1.tflite
!curl -L https://github.com/tensorflow/tflite-support/raw/master/tensorflow_lite_support/metadata/python/tests/testdata/object_detector/labelmap.txt -o ssd_mobilenet_labels.txt

Step 3: Create metadata writer and populate.

In [ ]:
ObjectDetectorWriter = object_detector.MetadataWriter
_MODEL_PATH = "ssd_mobilenet_v1.tflite"
# Task Library expects label files that are in the same format as the one below.
_LABEL_FILE = "ssd_mobilenet_labels.txt"
_SAVE_TO_PATH = "ssd_mobilenet_v1_metadata.tflite"
# Normalization parameters are required when reprocessing the image. It is
# optional if the image pixel values are in the range of [0, 255] and the input
# tensor is quantized to uint8. See the introduction for normalization and
# quantization parameters below for more details.
# https://www.tensorflow.org/lite/models/convert/metadata#normalization_and_quantization_parameters)
_INPUT_NORM_MEAN = 127.5
_INPUT_NORM_STD = 127.5

# Create the metadata writer.
writer = ObjectDetectorWriter.create_for_inference(
    writer_utils.load_file(_MODEL_PATH), [_INPUT_NORM_MEAN], [_INPUT_NORM_STD],
    [_LABEL_FILE])

# Verify the metadata generated by the metadata writer.
print(writer.get_metadata_json())

# Populate the metadata into the model.
writer_utils.save_file(writer.populate(), _SAVE_TO_PATH)

<a name=image_segmenters></a>
### Image segmenters

See the [image segmenter model compatibility requirements](https://www.tensorflow.org/lite/inference_with_metadata/task_library/image_segmenter#model_compatibility_requirements) for more details about the supported model format.

Step 1: Import the required packages.

In [ ]:
# from tflite_support.metadata_writers import image_segmenter
# from tflite_support.metadata_writers import writer_utils

Step 2: Download the example image segmenter, [deeplabv3.tflite](https://github.com/tensorflow/tflite-support/blob/master/tensorflow_lite_support/metadata/python/tests/testdata/image_segmenter/deeplabv3.tflite), and the [label file](https://github.com/tensorflow/tflite-support/blob/master/tensorflow_lite_support/metadata/python/tests/testdata/image_segmenter/labelmap.txt).

In [ ]:
!curl -L https://github.com/tensorflow/tflite-support/raw/master/tensorflow_lite_support/metadata/python/tests/testdata/image_segmenter/deeplabv3.tflite -o deeplabv3.tflite
!curl -L https://github.com/tensorflow/tflite-support/raw/master/tensorflow_lite_support/metadata/python/tests/testdata/image_segmenter/labelmap.txt -o deeplabv3_labels.txt

Step 3: Create metadata writer and populate.

In [ ]:
ImageSegmenterWriter = image_segmenter.MetadataWriter
_MODEL_PATH = "deeplabv3.tflite"
# Task Library expects label files that are in the same format as the one below.
_LABEL_FILE = "deeplabv3_labels.txt"
_SAVE_TO_PATH = "deeplabv3_metadata.tflite"
# Normalization parameters are required when reprocessing the image. It is
# optional if the image pixel values are in the range of [0, 255] and the input
# tensor is quantized to uint8. See the introduction for normalization and
# quantization parameters below for more details.
# https://www.tensorflow.org/lite/models/convert/metadata#normalization_and_quantization_parameters)
_INPUT_NORM_MEAN = 127.5
_INPUT_NORM_STD = 127.5

# Create the metadata writer.
writer = ImageSegmenterWriter.create_for_inference(
    writer_utils.load_file(_MODEL_PATH), [_INPUT_NORM_MEAN], [_INPUT_NORM_STD],
    [_LABEL_FILE])

# Verify the metadata generated by metadata writer.
print(writer.get_metadata_json())

# Populate the metadata into the model.
writer_utils.save_file(writer.populate(), _SAVE_TO_PATH)

<a name=nl_classifiers></a>
###Natural language classifiers

See the [natural language classifier model compatibility requirements](https://www.tensorflow.org/lite/inference_with_metadata/task_library/nl_classifier#model_compatibility_requirements) for more details about the supported model format.

Step 1: Import the required packages.

In [ ]:
# from tflite_support.metadata_writers import nl_classifier
# from tflite_support.metadata_writers import metadata_info
# from tflite_support.metadata_writers import writer_utils

Step 2: Download the example natural language classifier, [movie_review.tflite](https://github.com/tensorflow/tflite-support/blob/master/tensorflow_lite_support/metadata/python/tests/testdata/nl_classifier/movie_review.tflite), the [label file](https://github.com/tensorflow/tflite-support/blob/master/tensorflow_lite_support/metadata/python/tests/testdata/nl_classifier/labels.txt), and the [vocab file](https://storage.googleapis.com/download.tensorflow.org/models/tflite_support/nl_classifier/vocab.txt).

In [ ]:
!curl -L https://github.com/tensorflow/tflite-support/raw/master/tensorflow_lite_support/metadata/python/tests/testdata/nl_classifier/movie_review.tflite -o movie_review.tflite
!curl -L https://github.com/tensorflow/tflite-support/raw/master/tensorflow_lite_support/metadata/python/tests/testdata/nl_classifier/labels.txt -o movie_review_labels.txt
!curl -L https://storage.googleapis.com/download.tensorflow.org/models/tflite_support/nl_classifier/vocab.txt -o movie_review_vocab.txt

Step 3: Create metadata writer and populate.

In [ ]:
NLClassifierWriter = nl_classifier.MetadataWriter
_MODEL_PATH = "movie_review.tflite"
# Task Library expects label files and vocab files that are in the same formats
# as the ones below.
_LABEL_FILE = "movie_review_labels.txt"
_VOCAB_FILE = "movie_review_vocab.txt"
# NLClassifier supports tokenize input string using the regex tokenizer. See
# more details about how to set up RegexTokenizer below:
# https://github.com/tensorflow/tflite-support/blob/master/tensorflow_lite_support/metadata/python/metadata_writers/metadata_info.py#L130
_DELIM_REGEX_PATTERN = r"[^\w\']+"
_SAVE_TO_PATH = "moview_review_metadata.tflite"

# Create the metadata writer.
writer = nl_classifier.MetadataWriter.create_for_inference(
    writer_utils.load_file(_MODEL_PATH),
    metadata_info.RegexTokenizerMd(_DELIM_REGEX_PATTERN, _VOCAB_FILE),
    [_LABEL_FILE])

# Verify the metadata generated by metadata writer.
print(writer.get_metadata_json())

# Populate the metadata into the model.
writer_utils.save_file(writer.populate(), _SAVE_TO_PATH)

<a name=audio_classifiers></a>
### Audio classifiers

See the [audio classifier model compatibility requirements](https://www.tensorflow.org/lite/inference_with_metadata/task_library/audio_classifier#model_compatibility_requirements) for more details about the supported model format.

Step 1: Import the required packages.

In [ ]:
# from tflite_support.metadata_writers import audio_classifier
# from tflite_support.metadata_writers import metadata_info
# from tflite_support.metadata_writers import writer_utils

Step 2: Download the example audio classifier, [yamnet.tflite](https://github.com/tensorflow/tflite-support/blob/master/tensorflow_lite_support/metadata/python/tests/testdata/audio_classifier/yamnet_wavin_quantized_mel_relu6.tflite), and the [label file](https://github.com/tensorflow/tflite-support/blob/master/tensorflow_lite_support/metadata/python/tests/testdata/audio_classifier/yamnet_521_labels.txt).

In [ ]:
!curl -L https://github.com/tensorflow/tflite-support/raw/master/tensorflow_lite_support/metadata/python/tests/testdata/audio_classifier/yamnet_wavin_quantized_mel_relu6.tflite -o yamnet.tflite
!curl -L https://github.com/tensorflow/tflite-support/raw/master/tensorflow_lite_support/metadata/python/tests/testdata/audio_classifier/yamnet_521_labels.txt -o yamnet_labels.txt


Step 3: Create metadata writer and populate.

In [ ]:
AudioClassifierWriter = audio_classifier.MetadataWriter
_MODEL_PATH = "yamnet.tflite"
# Task Library expects label files that are in the same format as the one below.
_LABEL_FILE = "yamnet_labels.txt"
# Expected sampling rate of the input audio buffer.
_SAMPLE_RATE = 16000
# Expected number of channels of the input audio buffer. Note that Task library only
# supports single channel so far.
_CHANNELS = 1
_SAVE_TO_PATH = "yamnet_metadata.tflite"

# Create the metadata writer.
writer = AudioClassifierWriter.create_for_inference(
    writer_utils.load_file(_MODEL_PATH), _SAMPLE_RATE, _CHANNELS, [_LABEL_FILE])

# Verify the metadata generated by the metadata writer.
print(writer.get_metadata_json())

# Populate the metadata into the model.
writer_utils.save_file(writer.populate(), _SAVE_TO_PATH)

## Create Model Metadata with semantic information

You can fill in more descriptive information about the model and each tensor through the Metadata Writer API to help improve model understanding. It can be done through the 'create_from_metadata_info' method in each metadata writer. In general, you can fill in data through the parameters of 'create_from_metadata_info', i.e. `general_md`, `input_md`, and `output_md`. See the example below to create a rich Model Metadata for image classifiers.

Step 1: Import the required packages.

In [ ]:
# from tflite_support.metadata_writers import image_classifier
# from tflite_support.metadata_writers import metadata_info
# from tflite_support.metadata_writers import writer_utils
# from tflite_support import metadata_schema_py_generated as _metadata_fb

Step 2: Download the example image classifier, [mobilenet_v2_1.0_224.tflite](https://github.com/tensorflow/tflite-support/raw/master/tensorflow_lite_support/metadata/python/tests/testdata/image_classifier/mobilenet_v2_1.0_224.tflite), and the [label file](https://github.com/tensorflow/tflite-support/raw/master/tensorflow_lite_support/metadata/python/tests/testdata/image_classifier/labels.txt).

In [ ]:
!curl -L https://github.com/tensorflow/tflite-support/raw/master/tensorflow_lite_support/metadata/python/tests/testdata/image_classifier/mobilenet_v2_1.0_224.tflite -o mobilenet_v2_1.0_224.tflite
!curl -L https://github.com/tensorflow/tflite-support/raw/master/tensorflow_lite_support/metadata/python/tests/testdata/image_classifier/labels.txt -o mobilenet_labels.txt

Step 3: Create model and tensor information.

In [ ]:
# model_buffer = writer_utils.load_file("mobilenet_v2_1.0_224.tflite")

# # Create general model information.
# general_md = metadata_info.GeneralMd(
#     name="ImageClassifier",
#     version="v1",
#     description=("Identify the most prominent object in the image from a "
#                  "known set of categories."),
#     author="TensorFlow Lite",
#     licenses="Apache License. Version 2.0")

# # Create input tensor information.
# input_md = metadata_info.InputImageTensorMd(
#     name="input image",
#     description=("Input image to be classified. The expected image is "
#                  "128 x 128, with three channels (red, blue, and green) per "
#                  "pixel. Each element in the tensor is a value between min and "
#                  "max, where (per-channel) min is [0] and max is [255]."),
#     norm_mean=[127.5],
#     norm_std=[127.5],
#     color_space_type=_metadata_fb.ColorSpaceType.RGB,
#     tensor_type=writer_utils.get_input_tensor_types(model_buffer)[0])

# # Create output tensor information.
# output_md = metadata_info.ClassificationTensorMd(
#     name="probability",
#     description="Probabilities of the 1001 labels respectively.",
#     label_files=[
#         metadata_info.LabelFileMd(file_path="mobilenet_labels.txt",
#                                   locale="en")
#     ],
#     tensor_type=writer_utils.get_output_tensor_types(model_buffer)[0])

Step 4: Create metadata writer and populate.

In [ ]:
# ImageClassifierWriter = image_classifier.MetadataWriter
# # Create the metadata writer.
# writer = ImageClassifierWriter.create_from_metadata_info(
#     model_buffer, general_md, input_md, output_md)

# # Verify the metadata generated by the metadata writer.
# print(writer.get_metadata_json())

# # Populate the metadata into the model.
# writer_utils.save_file(writer.populate(), _SAVE_TO_PATH)

## Read the metadata populated to your model.

You can display the metadata and associated files in a TFLite model through the following code:

In [ ]:
# from tflite_support import metadata

# displayer = metadata.MetadataDisplayer.with_model_file("mobilenet_v2_1.0_224_metadata.tflite")
# print("Metadata populated:")
# print(displayer.get_metadata_json())

# print("Associated file(s) populated:")
# for file_name in displayer.get_packed_associated_file_list():
#   print("file name: ", file_name)
#   print("file content:")
#   print(displayer.get_associated_file_buffer(file_name))